# Patrón de Comportamiento: State

## Introducción
El patrón State permite que un objeto altere su comportamiento cuando su estado interno cambia, pareciendo que cambia su clase.

## Objetivos
- Comprender cómo encapsular comportamientos dependientes del estado.
- Identificar cuándo es útil el patrón State.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Cajero automático (ATM)**
Un cajero automático cambia su comportamiento según el estado: sin tarjeta, con tarjeta, con PIN correcto, sin fondos, etc.

**¿Dónde se usa en proyectos reales?**
En máquinas de estados, juegos, sistemas de workflow, etc.

## Sin patrón State (forma errónea)
El comportamiento se controla con múltiples condicionales.

In [1]:
class Cajero:
    def __init__(self):
        self.estado = 'sin_tarjeta'
    def insertar_tarjeta(self):
        if self.estado == 'sin_tarjeta':
            print('Tarjeta insertada')
            self.estado = 'con_tarjeta'
        else:
            print('Operación no permitida')

## Con patrón State (forma correcta)
Cada estado se implementa como una clase separada.

In [2]:
class Estado:
    def insertar_tarjeta(self, cajero):
        pass

class SinTarjeta(Estado):
    def insertar_tarjeta(self, cajero):
        print('Tarjeta insertada')
        cajero.estado = ConTarjeta()

class ConTarjeta(Estado):
    def insertar_tarjeta(self, cajero):
        print('Ya hay una tarjeta insertada')

class Cajero:
    def __init__(self):
        self.estado = SinTarjeta()
    def insertar_tarjeta(self):
        self.estado.insertar_tarjeta(self)

cajero = Cajero()
cajero.insertar_tarjeta()
cajero.insertar_tarjeta()

Tarjeta insertada
Ya hay una tarjeta insertada


## UML del patrón State
```plantuml
@startuml
class Cajero {
    + insertar_tarjeta()
}
class Estado {
    + insertar_tarjeta(cajero)
}
Estado <|-- SinTarjeta
Estado <|-- ConTarjeta
Cajero --> Estado
@enduml
```

## Otro ejemplo de la vida real: Máquina de estados de un pedido de e-commerce
**Contexto:** un pedido pasa por estados (pendiente, pagado, enviado, cancelado) y solo algunas transiciones son válidas según el estado actual — no se puede enviar un pedido que no se ha pagado, ni cancelar uno que ya fue entregado. A medida que se agregan más estados y reglas, un solo método con condicionales sobre strings se vuelve difícil de seguir y propenso a errores de tipeo.

### Sin patrón (forma errónea)
Cada operación revisa manualmente el string de estado actual antes de decidir qué hacer.

In [3]:
class Pedido:
    def __init__(self):
        self.estado = 'pendiente'
    def pagar(self):
        if self.estado == 'pendiente':
            print('Pago confirmado')
            self.estado = 'pagado'
        else:
            print(f'No se puede pagar un pedido en estado {self.estado}')
    def enviar(self):
        if self.estado == 'pagado':
            print('Pedido enviado')
            self.estado = 'enviado'
        else:
            print(f'No se puede enviar un pedido en estado {self.estado}')
    def cancelar(self):
        if self.estado in ('pendiente', 'pagado'):
            print('Pedido cancelado')
            self.estado = 'cancelado'
        else:
            print(f'No se puede cancelar un pedido en estado {self.estado}')

# Cada método repite la lista de estados válidos como strings sueltos: fácil de desalinear
pedido = Pedido()
pedido.pagar()
pedido.enviar()
pedido.pagar()

Pago confirmado
Pedido enviado
No se puede pagar un pedido en estado enviado


### Con patrón (forma correcta)
Cada estado es una clase que solo implementa las transiciones que le son válidas; las demás simplemente heredan el comportamiento "no permitido" de la clase base.

In [4]:
class EstadoPedido:
    def pagar(self, pedido):
        print('Operación no permitida en este estado')
    def enviar(self, pedido):
        print('Operación no permitida en este estado')
    def cancelar(self, pedido):
        print('Operación no permitida en este estado')

class Pendiente(EstadoPedido):
    def pagar(self, pedido):
        print('Pago confirmado')
        pedido.estado = Pagado()
    def cancelar(self, pedido):
        print('Pedido cancelado')
        pedido.estado = Cancelado()

class Pagado(EstadoPedido):
    def enviar(self, pedido):
        print('Pedido enviado')
        pedido.estado = Enviado()
    def cancelar(self, pedido):
        print('Pedido cancelado (con reembolso)')
        pedido.estado = Cancelado()

class Enviado(EstadoPedido):
    pass  # no permite pagar de nuevo, reenviar ni cancelar

class Cancelado(EstadoPedido):
    pass


class Pedido:
    def __init__(self):
        self.estado = Pendiente()
    def pagar(self):
        self.estado.pagar(self)
    def enviar(self):
        self.estado.enviar(self)
    def cancelar(self):
        self.estado.cancelar(self)


pedido = Pedido()
pedido.pagar()
pedido.enviar()
pedido.pagar()  # ya está enviado: no debería permitir pagar de nuevo

Pago confirmado
Pedido enviado
Operación no permitida en este estado


### UML del ejemplo de estados de pedido
```plantuml
@startuml
class Pedido {
    + pagar()
    + enviar()
    + cancelar()
}
abstract class EstadoPedido {
    + pagar(pedido)
    + enviar(pedido)
    + cancelar(pedido)
}
class Pendiente
class Pagado
class Enviado
class Cancelado
EstadoPedido <|-- Pendiente
EstadoPedido <|-- Pagado
EstadoPedido <|-- Enviado
EstadoPedido <|-- Cancelado
Pedido --> EstadoPedido
@enduml
```

### ¿Dónde más se usa State?
- **Estados de un pedido/orden:** exactamente este ejemplo — Amazon, Mercado Libre y cualquier e-commerce modelan el ciclo de vida del pedido así.
- **Cajeros automáticos:** el ejemplo con el que abre este notebook — sin tarjeta, con tarjeta, PIN validado, sin fondos.
- **Semáforos y controladores de tráfico:** el comportamiento (qué luces se encienden) cambia completamente según el estado actual.
- **Reproductores multimedia:** un reproductor se comporta distinto si está reproduciendo, pausado o detenido (el botón "play/pause" hace algo distinto según el estado).
- **Flujos de aprobación (workflows):** un documento o solicitud que pasa por "borrador → en revisión → aprobado → publicado", donde cada estado permite solo ciertas acciones.

**Ejercicio de reflexión:** ¿por qué en la versión "con patrón" no fue necesario un `if/elif` en ningún método de `Pedido`? ¿Dónde "se escondió" la lógica condicional que antes vivía en la clase `Pedido`?

## Actividad
Crea un sistema de semáforo donde el comportamiento cambie según el estado (rojo, amarillo, verde).

---
## Explicación de conceptos clave
- **Encapsulamiento de estados:** Cada estado es una clase diferente.
- **Eliminación de condicionales:** El comportamiento depende del estado actual.
- **Aplicación en la vida real:** Útil en máquinas de estados, juegos y sistemas de workflow.

## Conclusión
El patrón State es ideal para sistemas donde el comportamiento depende del estado interno del objeto.